# TCGA-LUAD Phase 8 — Reproducible biological interpretation

Uses the exact Phase 6 and Phase 7B derived inputs packaged in the project ZIP. The precomputed PDF report is supplied separately. Run the cells in order. The notebook regenerates descriptive figures and summary tables, not differential-expression tests.

In [ ]:
from google.colab import files
import zipfile, pathlib
uploaded=files.upload()
archive=next(k for k in uploaded if k.endswith(".zip"))
with zipfile.ZipFile(archive) as z: z.extractall("/content/phase8")
print("Extracted Phase 8 inputs")

In [ ]:
%pip -q install pandas numpy matplotlib scipy

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
P=Path("/content/phase8")
D=pd.read_csv(P/"input/annotated_dge.csv")
R=pd.read_csv(P/"input/gsea_Reactome.csv")
G=pd.read_csv(P/"input/gsea_GO_BP.csv")
M=pd.read_csv(P/"input/aligned_metadata.csv").set_index("sample_barcode")
V=pd.read_csv(P/"input/vst_matrix.csv.gz",index_col=0)
assert set(M.index)==set(V.columns)
print("DGE:",D.shape,"VST:",V.shape,"samples:",M.sample_type.value_counts().to_dict())

In [ ]:
R["NES"]=pd.to_numeric(R.NES,errors="coerce")
R["FDR q-val"]=pd.to_numeric(R["FDR q-val"],errors="coerce")
sig=R[R["FDR q-val"]<.05]
plot=pd.concat([sig[sig.NES<0].nsmallest(8,"NES"),sig[sig.NES>0].nlargest(8,"NES")]).sort_values("NES")
plt.figure(figsize=(11,7))
plt.barh(plot.Term.str.slice(0,52),plot.NES,color=["#257e9a" if n<0 else "#c66b37" for n in plot.NES])
plt.axvline(0,color="black",lw=.8);plt.xlabel("Reactome GSEA NES");plt.tight_layout();plt.show()

In [ ]:
candidate=["COL1A1","COL1A2","COL3A1","POSTN","MMP1","MMP7","MKI67","TOP2A","AGER","SFTPA1"]
selected=D[D.gene_name.isin(candidate)].sort_values("padj").drop_duplicates("gene_name")
display(selected[["gene_name","log2FoldChange","padj","baseMean"]])
for _,g in selected.head(6).iterrows():
 if g.gene_id not in V.index: continue
 expr=V.loc[g.gene_id];groups=[expr[M.loc[V.columns,"sample_type"]==name].values for name in ["Solid Tissue Normal","Primary Tumor"]]
 plt.figure(figsize=(4,3));plt.boxplot(groups,tick_labels=["Normal","Tumor"]);plt.title(g.gene_name);plt.ylabel("VST expression");plt.show()

In [ ]:
counts=Counter()
for s in sig.Lead_genes.dropna():
 for g in set(str(s).split(";")):
  if g.strip():counts[g.strip()]+=1
freq=pd.DataFrame(counts.most_common(),columns=["gene_symbol","n_significant_reactome_sets"])
display(freq.head(20))
freq.to_csv("/content/reactome_leading_edge_frequency.csv",index=False)
files.download("/content/reactome_leading_edge_frequency.csv")

## Interpretation notes
- NES direction follows the original tumor-vs-normal DESeq2 contrast.
- Leading-edge frequency is inflated by overlapping gene sets and is not a biomarker ranking.
- Bulk expression cannot identify cell-of-origin or prove signaling activity.
- LOSO was intentionally skipped. Independent validation remains pending.